# Multimodal Data Quality Scoring + Best-Use Recommendation

This notebook evaluates **tabular**, **image**, and **text** data quality using common industry dimensions:

- Completeness
- Validity
- Consistency
- Uniqueness / Duplicates
- Representativeness
- (Optional) Balance and Timeliness

It outputs:

1. A **0-100 quality score** by dimension and overall.
2. A **quality tier** (Excellent / Good / Fair / Poor).
3. The **best use case(s)** for the dataset based on strengths/weaknesses.
4. A prioritized list of **fix actions** for weak areas.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union
import json
import math
import re
import string

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tif", ".tiff"}
TABULAR_EXTENSIONS = {".csv", ".parquet", ".pq", ".json", ".xlsx", ".xls"}
TEXT_EXTENSIONS = {".txt", ".md", ".log"}

PII_NAME_HINTS = {
    "name", "first_name", "last_name", "full_name", "email", "phone", "mobile",
    "ssn", "social", "dob", "birth", "address", "zip", "postal", "passport",
    "license", "account", "iban", "credit", "card"
}

PII_REGEX = {
    "email": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    "phone": r"(?:\+?1[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?){2}\d{4}",
    "ssn": r"\b\d{3}-\d{2}-\d{4}\b",
    "credit_card": r"\b(?:\d[ -]*?){13,16}\b",
}


def _clamp(x: float, low: float = 0.0, high: float = 100.0) -> float:
    return float(max(low, min(high, x)))


def _safe_mean(values: Sequence[float]) -> float:
    vals = [float(v) for v in values if v is not None and not pd.isna(v)]
    return float(np.mean(vals)) if vals else 0.0


def _weighted_score(scores: Dict[str, float], weights: Dict[str, float]) -> float:
    available = {k: v for k, v in scores.items() if v is not None and not pd.isna(v)}
    if not available:
        return 0.0
    weight_sum = sum(weights.get(k, 1.0) for k in available)
    if weight_sum <= 0:
        return _safe_mean(list(available.values()))
    return float(sum(available[k] * weights.get(k, 1.0) for k in available) / weight_sum)


def _quality_tier(score: float) -> str:
    if score >= 85:
        return "Excellent"
    if score >= 70:
        return "Good"
    if score >= 55:
        return "Fair"
    return "Poor"


def _top_bottom(scores: Dict[str, float], n: int = 3) -> Tuple[List[Tuple[str, float]], List[Tuple[str, float]]]:
    ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return ordered[:n], ordered[-n:]


def _json_default(obj: Any):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if isinstance(obj, Path):
        return str(obj)
    return str(obj)


def _detect_pii_in_series(series: pd.Series, sample_size: int = 500) -> List[str]:
    sample = series.dropna().astype(str).head(sample_size)
    hits = []
    for label, pattern in PII_REGEX.items():
        try:
            if sample.str.contains(pattern, regex=True).any():
                hits.append(label)
        except re.error:
            continue
    return sorted(set(hits))


def _token_type(value: Any) -> str:
    text = str(value).strip()
    if not text:
        return "empty"
    if re.fullmatch(r"[-+]?\d+(?:\.\d+)?", text):
        return "number"
    if re.fullmatch(r"\d{4}[-/]\d{1,2}[-/]\d{1,2}(?:.*)?", text):
        return "date"
    if re.fullmatch(r"(?i)(true|false|yes|no|y|n)", text):
        return "bool"
    return "text"


In [ ]:
def detect_modality(data: Any, modality: str = "auto") -> str:
    if modality and modality.lower() != "auto":
        return modality.lower()

    if isinstance(data, pd.DataFrame):
        return "tabular"
    if isinstance(data, np.ndarray) and data.ndim in (2, 3):
        return "image"

    if isinstance(data, (list, tuple)) and len(data) > 0:
        if all(isinstance(x, np.ndarray) for x in data):
            return "image"
        if all(isinstance(x, str) for x in data):
            path_count = sum(Path(x).exists() for x in data)
            if path_count == 0:
                return "text"
            path_objs = [Path(x) for x in data if Path(x).exists()]
            if path_objs and all(p.suffix.lower() in IMAGE_EXTENSIONS for p in path_objs):
                return "image"
            if path_objs and all(p.suffix.lower() in TABULAR_EXTENSIONS for p in path_objs):
                return "tabular"
            if path_objs and all(p.suffix.lower() in TEXT_EXTENSIONS for p in path_objs):
                return "text"

    if isinstance(data, (str, Path)):
        p = Path(data)
        if p.exists():
            if p.is_file():
                ext = p.suffix.lower()
                if ext in IMAGE_EXTENSIONS:
                    return "image"
                if ext in TABULAR_EXTENSIONS:
                    return "tabular"
                if ext in TEXT_EXTENSIONS:
                    return "text"
            if p.is_dir():
                files = [f for f in p.iterdir() if f.is_file()]
                img_n = sum(f.suffix.lower() in IMAGE_EXTENSIONS for f in files)
                tab_n = sum(f.suffix.lower() in TABULAR_EXTENSIONS for f in files)
                txt_n = sum(f.suffix.lower() in TEXT_EXTENSIONS for f in files)
                if img_n >= max(tab_n, txt_n) and img_n > 0:
                    return "image"
                if tab_n >= max(img_n, txt_n) and tab_n > 0:
                    return "tabular"
                if txt_n > 0:
                    return "text"

    return "tabular"


def _load_tabular(data: Any) -> pd.DataFrame:
    if isinstance(data, pd.DataFrame):
        return data.copy()
    if isinstance(data, (str, Path)):
        path = Path(data)
        if not path.exists() or not path.is_file():
            raise ValueError(f"Tabular path not found: {path}")
        ext = path.suffix.lower()
        if ext == ".csv":
            return pd.read_csv(path)
        if ext in {".parquet", ".pq"}:
            return pd.read_parquet(path)
        if ext in {".xlsx", ".xls"}:
            return pd.read_excel(path)
        if ext == ".json":
            return pd.read_json(path)
    raise ValueError("Unsupported tabular input. Use a DataFrame or a table file path.")


def _collect_image_candidates(data: Any) -> List[Tuple[str, Any]]:
    candidates: List[Tuple[str, Any]] = []

    if isinstance(data, np.ndarray):
        return [("array_0", data)]

    if isinstance(data, (str, Path)):
        p = Path(data)
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
            return [(str(p), p)]
        if p.is_dir():
            for f in sorted(p.rglob("*")):
                if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS:
                    candidates.append((str(f), f))
            return candidates

    if isinstance(data, (list, tuple)):
        for i, item in enumerate(data):
            if isinstance(item, np.ndarray):
                candidates.append((f"array_{i}", item))
            else:
                p = Path(item)
                if p.exists() and p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
                    candidates.append((str(p), p))
        return candidates

    raise ValueError("Unsupported image input. Use path(s), directory, or numpy array(s).")


def _load_text_items(data: Any) -> List[str]:
    texts: List[str] = []

    if isinstance(data, pd.Series):
        return [str(x) for x in data.tolist()]

    if isinstance(data, str) and not Path(data).exists():
        return [data]

    if isinstance(data, (str, Path)):
        p = Path(data)
        if p.is_file():
            return [p.read_text(errors="ignore")]
        if p.is_dir():
            for f in sorted(p.rglob("*")):
                if f.is_file() and f.suffix.lower() in TEXT_EXTENSIONS:
                    texts.append(f.read_text(errors="ignore"))
            return texts

    if isinstance(data, (list, tuple)):
        for item in data:
            if isinstance(item, (str, Path)) and Path(item).exists() and Path(item).is_file():
                texts.append(Path(item).read_text(errors="ignore"))
            else:
                texts.append(str(item))
        return texts

    raise ValueError("Unsupported text input. Use a text file/dir, Series, list, or string.")


In [ ]:
def evaluate_tabular(
    data: Any,
    target_col: Optional[str] = None,
    datetime_cols: Optional[List[str]] = None,
) -> Dict[str, Any]:
    df = _load_tabular(data)
    if df.empty:
        raise ValueError("Tabular dataset is empty.")

    n_rows, n_cols = df.shape
    total_cells = max(1, n_rows * n_cols)

    missing_ratio = float(df.isna().sum().sum() / total_cells)
    completeness = 100.0 * (1.0 - missing_ratio)

    duplicate_ratio = float(df.duplicated().mean()) if n_rows > 1 else 0.0
    integrity = 100.0 * (1.0 - duplicate_ratio)

    constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    constant_ratio = len(constant_cols) / max(1, n_cols)

    numeric_cols = list(df.select_dtypes(include=[np.number]).columns)
    validity_parts: List[float] = []

    if numeric_cols:
        num_data = df[numeric_cols].to_numpy(dtype=float, copy=False)
        inf_mask = np.isinf(num_data)
        inf_ratio = float(inf_mask.sum() / max(1, num_data.size))
        validity_parts.append(1.0 - inf_ratio)

    inferred_dt_cols = []
    if datetime_cols is not None:
        inferred_dt_cols = [c for c in datetime_cols if c in df.columns]
    else:
        for c in df.columns:
            if pd.api.types.is_datetime64_any_dtype(df[c]) or any(k in c.lower() for k in ["date", "time", "timestamp"]):
                inferred_dt_cols.append(c)

    dt_parse_fail_ratio: Dict[str, float] = {}
    parsed_dt: Dict[str, pd.Series] = {}
    for c in inferred_dt_cols:
        parsed = pd.to_datetime(df[c], errors="coerce", utc=True)
        parsed_dt[c] = parsed
        non_null = df[c].notna().sum()
        if non_null > 0:
            fail_ratio = float((parsed.isna() & df[c].notna()).sum() / non_null)
            dt_parse_fail_ratio[c] = fail_ratio
            validity_parts.append(1.0 - fail_ratio)

    validity = 100.0 * _safe_mean(validity_parts) if validity_parts else 100.0

    consistency_components: List[float] = []
    for c in df.columns:
        col = df[c].dropna()
        if col.empty:
            continue
        if (
            pd.api.types.is_numeric_dtype(df[c])
            or pd.api.types.is_datetime64_any_dtype(df[c])
            or pd.api.types.is_bool_dtype(df[c])
        ):
            consistency_components.append(1.0)
            continue
        sample = col.astype(str).head(1000)
        token_types = {_token_type(v) for v in sample}
        if len(token_types) <= 1:
            consistency_components.append(1.0)
        elif len(token_types) == 2:
            consistency_components.append(0.8)
        elif len(token_types) == 3:
            consistency_components.append(0.6)
        else:
            consistency_components.append(0.4)

    consistency = 100.0 * _safe_mean(consistency_components) if consistency_components else 100.0

    uniqueness = 100.0 * _safe_mean([1.0 - duplicate_ratio, 1.0 - constant_ratio])

    outlier_ratios: Dict[str, float] = {}
    for c in numeric_cols:
        s = pd.to_numeric(df[c], errors="coerce").dropna()
        if len(s) < 10:
            continue
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:
            outlier_ratios[c] = 0.0
            continue
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_ratios[c] = float(((s < lower) | (s > upper)).mean())

    outliers = 100.0 * (1.0 - _safe_mean(list(outlier_ratios.values()))) if outlier_ratios else 100.0

    balance = None
    balance_details: Dict[str, Any] = {}
    if target_col and target_col in df.columns:
        vc = df[target_col].dropna().value_counts()
        if len(vc) > 1:
            ratio = float(vc.min() / vc.max())
            probs = (vc / vc.sum()).to_numpy(dtype=float)
            entropy = float(-(probs * np.log2(probs + 1e-12)).sum() / np.log2(len(probs)))
            balance = 100.0 * _safe_mean([ratio, entropy])
            balance_details = {
                "minority_majority_ratio": ratio,
                "normalized_entropy": entropy,
                "class_counts": vc.to_dict(),
            }
        elif len(vc) == 1:
            balance = 30.0
            balance_details = {"class_counts": vc.to_dict(), "note": "Only one class present"}

    timeliness = None
    freshness_by_col: Dict[str, float] = {}
    if inferred_dt_cols:
        scores = []
        now = pd.Timestamp.now(tz="UTC")
        for c in inferred_dt_cols:
            parsed = parsed_dt.get(c)
            if parsed is None:
                parsed = pd.to_datetime(df[c], errors="coerce", utc=True)
            parsed = parsed.dropna()
            if parsed.empty:
                continue
            age_days = max(0.0, (now - parsed.max()).total_seconds() / 86400.0)
            freshness = max(0.0, 1.0 - age_days / (365.0 * 3.0))
            freshness_by_col[c] = freshness * 100.0
            scores.append(freshness * 100.0)
            if len(parsed) > 1:
                order_score = float((parsed.diff().dropna() >= pd.Timedelta(0)).mean() * 100.0)
                scores.append(order_score)
        if scores:
            timeliness = _safe_mean(scores)

    sample_size_score = min(100.0, math.log10(n_rows + 1) / math.log10(20000 + 1) * 100.0)
    coverage_score = (1.0 - constant_ratio) * 100.0
    filled_col_ratio = float((df.notna().mean() >= 0.7).mean())
    representativeness = _safe_mean([sample_size_score, coverage_score, filled_col_ratio * 100.0])

    pii_hits: Dict[str, List[str]] = {}
    for c in df.columns:
        col_hits: List[str] = []
        if c.lower() in PII_NAME_HINTS or any(hint in c.lower() for hint in PII_NAME_HINTS):
            col_hits.append("name_hint")
        if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c]):
            col_hits.extend(_detect_pii_in_series(df[c]))
        if col_hits:
            pii_hits[c] = sorted(set(col_hits))

    dimension_scores: Dict[str, float] = {
        "completeness": _clamp(completeness),
        "validity": _clamp(validity),
        "consistency": _clamp(consistency),
        "uniqueness": _clamp(uniqueness),
        "integrity": _clamp(integrity),
        "outliers": _clamp(outliers),
        "representativeness": _clamp(representativeness),
    }
    if balance is not None:
        dimension_scores["balance"] = _clamp(balance)
    if timeliness is not None:
        dimension_scores["timeliness"] = _clamp(timeliness)

    weights = {
        "completeness": 0.20,
        "validity": 0.15,
        "consistency": 0.10,
        "uniqueness": 0.10,
        "integrity": 0.10,
        "outliers": 0.10,
        "representativeness": 0.10,
        "balance": 0.10,
        "timeliness": 0.05,
    }
    overall = _weighted_score(dimension_scores, weights)
    strengths, weaknesses = _top_bottom(dimension_scores, n=3)

    return {
        "modality": "tabular",
        "n_rows": int(n_rows),
        "n_cols": int(n_cols),
        "overall_score": round(_clamp(overall), 2),
        "quality_tier": _quality_tier(overall),
        "dimension_scores": {k: round(v, 2) for k, v in dimension_scores.items()},
        "strengths": [{"metric": k, "score": round(v, 2)} for k, v in strengths],
        "weaknesses": [{"metric": k, "score": round(v, 2)} for k, v in weaknesses],
        "details": {
            "missing_ratio": round(missing_ratio, 4),
            "duplicate_ratio": round(duplicate_ratio, 4),
            "constant_columns": constant_cols,
            "numeric_columns": numeric_cols,
            "datetime_parse_fail_ratio": {k: round(v, 4) for k, v in dt_parse_fail_ratio.items()},
            "outlier_ratio_by_column": {k: round(v, 4) for k, v in outlier_ratios.items()},
            "freshness_score_by_datetime_column": {k: round(v, 2) for k, v in freshness_by_col.items()},
            "potential_pii_columns": pii_hits,
            "balance": balance_details,
        },
    }


In [ ]:
def _average_hash(image: Image.Image, hash_size: int = 8) -> str:
    gray = image.convert("L").resize((hash_size, hash_size), Image.Resampling.BILINEAR)
    arr = np.asarray(gray, dtype=np.float32)
    bits = arr > arr.mean()
    return "".join("1" if b else "0" for b in bits.flatten())


def _image_sharpness_score(gray: np.ndarray) -> float:
    gy, gx = np.gradient(gray)
    sharpness = float(np.var(np.hypot(gx, gy)))
    # Saturating normalization; 0 stays 0, high detail trends toward 100.
    return _clamp(100.0 * (sharpness / (sharpness + 80.0)))


def evaluate_images(
    data: Any,
    labels: Optional[Dict[str, Any]] = None,
    min_resolution: Tuple[int, int] = (224, 224),
) -> Dict[str, Any]:
    candidates = _collect_image_candidates(data)
    if not candidates:
        raise ValueError("No image files found in input.")

    records: List[Dict[str, Any]] = []
    corrupted: List[str] = []

    for name, item in candidates:
        try:
            if isinstance(item, np.ndarray):
                arr = np.asarray(item)
                if arr.ndim == 2:
                    img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8), mode="L").convert("RGB")
                elif arr.ndim == 3:
                    img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8)).convert("RGB")
                else:
                    raise ValueError("Unsupported image array shape")
            else:
                with Image.open(item) as im:
                    img = im.convert("RGB")

            rgb = np.asarray(img, dtype=np.uint8)
            gray = np.asarray(img.convert("L"), dtype=np.float32)
            h, w = gray.shape
            resolution = int(w * h)
            brightness = float(gray.mean())
            contrast = float(gray.std())
            sharpness_score = _image_sharpness_score(gray)
            aspect_ratio = float(w / max(1, h))
            img_hash = _average_hash(img)

            records.append({
                "id": name,
                "width": int(w),
                "height": int(h),
                "resolution": resolution,
                "brightness": brightness,
                "contrast": contrast,
                "sharpness_score": sharpness_score,
                "aspect_ratio": aspect_ratio,
                "hash": img_hash,
            })
        except (UnidentifiedImageError, OSError, ValueError):
            corrupted.append(name)

    total_n = len(candidates)
    valid_n = len(records)
    if valid_n == 0:
        raise ValueError("All images failed to load or decode.")

    integrity = 100.0 * (valid_n / total_n)

    resolutions = np.array([r["resolution"] for r in records], dtype=float)
    target_pixels = float(min_resolution[0] * min_resolution[1])
    resolution_score = float(np.mean(np.clip(resolutions / max(1.0, target_pixels), 0.0, 1.0)) * 100.0)

    sharpness = _safe_mean([r["sharpness_score"] for r in records])

    brightness_vals = np.array([r["brightness"] for r in records], dtype=float)
    contrast_vals = np.array([r["contrast"] for r in records], dtype=float)
    exposure_score = float(np.mean(np.clip(1.0 - (np.abs(brightness_vals - 128.0) / 128.0), 0.0, 1.0)) * 100.0)
    contrast_score = float(np.mean(np.clip((contrast_vals - 10.0) / 50.0, 0.0, 1.0)) * 100.0)

    hashes = [r["hash"] for r in records]
    unique_hash_ratio = len(set(hashes)) / max(1, len(hashes))
    deduplication = 100.0 * unique_hash_ratio

    aspect_ratios = np.array([r["aspect_ratio"] for r in records], dtype=float)
    if len(aspect_ratios) >= 4:
        q1, q3 = np.percentile(aspect_ratios, [25, 75])
        iqr = q3 - q1
        if iqr > 0:
            lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            aspect_outlier_ratio = float(((aspect_ratios < lower) | (aspect_ratios > upper)).mean())
        else:
            aspect_outlier_ratio = 0.0
    else:
        aspect_outlier_ratio = 0.0
    consistency = 100.0 * (1.0 - aspect_outlier_ratio)

    sample_size_score = min(100.0, math.log10(valid_n + 1) / math.log10(5000 + 1) * 100.0)
    representativeness = _safe_mean([sample_size_score, unique_hash_ratio * 100.0, consistency])

    balance = None
    balance_details = {}
    if labels:
        label_values = [labels.get(r["id"]) for r in records if labels.get(r["id"]) is not None]
        if label_values:
            vc = pd.Series(label_values).value_counts()
            if len(vc) > 1:
                ratio = float(vc.min() / vc.max())
                probs = (vc / vc.sum()).to_numpy(dtype=float)
                entropy = float(-(probs * np.log2(probs + 1e-12)).sum() / np.log2(len(probs)))
                balance = 100.0 * _safe_mean([ratio, entropy])
                balance_details = {"class_counts": vc.to_dict(), "minority_majority_ratio": ratio, "normalized_entropy": entropy}

    dimension_scores = {
        "integrity": _clamp(integrity),
        "resolution": _clamp(resolution_score),
        "sharpness": _clamp(sharpness),
        "exposure": _clamp(exposure_score),
        "contrast": _clamp(contrast_score),
        "deduplication": _clamp(deduplication),
        "consistency": _clamp(consistency),
        "representativeness": _clamp(representativeness),
    }
    if balance is not None:
        dimension_scores["balance"] = _clamp(balance)

    weights = {
        "integrity": 0.22,
        "resolution": 0.15,
        "sharpness": 0.15,
        "exposure": 0.10,
        "contrast": 0.10,
        "deduplication": 0.10,
        "consistency": 0.08,
        "representativeness": 0.10,
        "balance": 0.10,
    }
    overall = _weighted_score(dimension_scores, weights)
    strengths, weaknesses = _top_bottom(dimension_scores, n=3)

    return {
        "modality": "image",
        "n_images": total_n,
        "n_valid_images": valid_n,
        "overall_score": round(_clamp(overall), 2),
        "quality_tier": _quality_tier(overall),
        "dimension_scores": {k: round(v, 2) for k, v in dimension_scores.items()},
        "strengths": [{"metric": k, "score": round(v, 2)} for k, v in strengths],
        "weaknesses": [{"metric": k, "score": round(v, 2)} for k, v in weaknesses],
        "details": {
            "corrupted_images": corrupted,
            "min_resolution_target": {"width": min_resolution[0], "height": min_resolution[1]},
            "duplicate_ratio": round(1.0 - unique_hash_ratio, 4),
            "balance": balance_details,
            "sample_image_stats": records[:10],
        },
    }


In [ ]:
def evaluate_text(data: Any) -> Dict[str, Any]:
    texts = _load_text_items(data)
    if not texts:
        raise ValueError("No text items found in input.")

    n_items = len(texts)
    stripped = [t.strip() for t in texts]
    non_empty = [t for t in stripped if t]

    completeness = 100.0 * (len(non_empty) / max(1, n_items))

    normalized = [re.sub(r"\s+", " ", t.lower()) for t in non_empty]
    unique_ratio = len(set(normalized)) / max(1, len(normalized))
    uniqueness = 100.0 * unique_ratio

    token_counts = [len(re.findall(r"\w+", t)) for t in non_empty]
    if token_counts:
        token_arr = np.array(token_counts, dtype=float)
        short_ratio = float((token_arr < 5).mean())
        long_ratio = float((token_arr > 1500).mean())
        length_quality = 100.0 * (1.0 - _safe_mean([short_ratio, long_ratio]))
        variability = float(np.std(token_arr) / max(1.0, np.mean(token_arr)))
        consistency = 100.0 * (1.0 - min(1.0, variability / 2.5))
    else:
        length_quality = 0.0
        consistency = 0.0

    noise_ratios = []
    alpha_ratios = []
    pii_hits = {"email": 0, "phone": 0, "ssn": 0, "credit_card": 0}

    for t in non_empty:
        total = max(1, len(t))
        non_print = sum(ch not in string.printable for ch in t)
        alpha = sum(ch.isalpha() for ch in t)
        noise_ratios.append(non_print / total)
        alpha_ratios.append(alpha / total)

        for label, pattern in PII_REGEX.items():
            try:
                if re.search(pattern, t):
                    pii_hits[label] += 1
            except re.error:
                continue

    validity = 100.0 * (1.0 - min(1.0, _safe_mean(noise_ratios) * 10.0))
    language_consistency = 100.0 * _safe_mean(alpha_ratios) if alpha_ratios else 0.0

    tokens = []
    for t in non_empty:
        tokens.extend(re.findall(r"[A-Za-z']+", t.lower()))
    vocab_diversity = len(set(tokens)) / max(1, len(tokens))
    sample_size_score = min(100.0, math.log10(n_items + 1) / math.log10(2000 + 1) * 100.0)
    representativeness = _safe_mean([sample_size_score, min(100.0, vocab_diversity * 400.0), uniqueness])

    dimension_scores = {
        "completeness": _clamp(completeness),
        "validity": _clamp(validity),
        "consistency": _clamp(consistency),
        "uniqueness": _clamp(uniqueness),
        "length_quality": _clamp(length_quality),
        "language_consistency": _clamp(language_consistency),
        "representativeness": _clamp(representativeness),
    }

    weights = {
        "completeness": 0.16,
        "validity": 0.16,
        "consistency": 0.12,
        "uniqueness": 0.16,
        "length_quality": 0.12,
        "language_consistency": 0.12,
        "representativeness": 0.16,
    }

    overall = _weighted_score(dimension_scores, weights)
    strengths, weaknesses = _top_bottom(dimension_scores, n=3)

    pii_risk = "low"
    if any(v > 0 for v in pii_hits.values()):
        pii_risk = "medium"
    if sum(v > 0 for v in pii_hits.values()) >= 2:
        pii_risk = "high"

    return {
        "modality": "text",
        "n_documents": int(n_items),
        "overall_score": round(_clamp(overall), 2),
        "quality_tier": _quality_tier(overall),
        "dimension_scores": {k: round(v, 2) for k, v in dimension_scores.items()},
        "strengths": [{"metric": k, "score": round(v, 2)} for k, v in strengths],
        "weaknesses": [{"metric": k, "score": round(v, 2)} for k, v in weaknesses],
        "details": {
            "token_count_summary": {
                "min": int(min(token_counts)) if token_counts else 0,
                "median": float(np.median(token_counts)) if token_counts else 0,
                "max": int(max(token_counts)) if token_counts else 0,
            },
            "vocab_diversity": round(vocab_diversity, 4),
            "pii_pattern_hits": pii_hits,
            "pii_risk": pii_risk,
        },
    }


In [ ]:
def _recommend_use_cases(modality: str, scores: Dict[str, float], overall: float) -> Tuple[List[str], List[str]]:
    recommendations: List[str] = []
    cautions: List[str] = []

    if modality == "tabular":
        if overall >= 85 and scores.get("completeness", 0) >= 85 and scores.get("validity", 0) >= 80:
            recommendations.extend([
                "Production-grade supervised ML training.",
                "High-confidence BI dashboards and forecasting.",
                "Feature store and model-monitoring baselines.",
            ])
        elif overall >= 70:
            recommendations.extend([
                "Baseline model training and iteration.",
                "Decision analytics with targeted cleaning.",
                "Clustering and segmentation analysis.",
            ])
        elif overall >= 55:
            recommendations.extend([
                "Exploratory data analysis and hypothesis testing.",
                "Unsupervised anomaly detection after light cleanup.",
                "Data quality remediation pipeline input.",
            ])
        else:
            recommendations.extend([
                "Data profiling only (not recommended for training yet).",
                "Quality-improvement backlog and schema hardening.",
            ])

        if scores.get("balance", 100) < 60:
            cautions.append("Class imbalance is high; reweighting/resampling is recommended before supervised training.")
        if scores.get("outliers", 100) < 60:
            cautions.append("Outlier rate is elevated; robust scaling and outlier policy should be applied.")
        if scores.get("timeliness", 100) < 50:
            cautions.append("Data may be stale for real-time or near-real-time decisions.")

    elif modality == "image":
        if overall >= 85 and scores.get("integrity", 0) >= 90 and scores.get("resolution", 0) >= 80 and scores.get("sharpness", 0) >= 75:
            recommendations.extend([
                "Computer vision model training (classification/detection).",
                "High-quality fine-tuning datasets for production.",
                "Benchmark and validation dataset creation.",
            ])
        elif overall >= 70:
            recommendations.extend([
                "Model prototyping and transfer learning.",
                "Visual analytics/search with preprocessing.",
                "Augmentation-backed training workflows.",
            ])
        elif overall >= 55:
            recommendations.extend([
                "Coarse classification tasks with heavy augmentation.",
                "QA and labeling workflow development.",
                "Pre-cleaning input for future training datasets.",
            ])
        else:
            recommendations.extend([
                "Manual review and data-cleaning only.",
                "Not recommended for direct training until quality improves.",
            ])

        if scores.get("deduplication", 100) < 70:
            cautions.append("Duplicate images are likely; run dedup before training.")
        if scores.get("sharpness", 100) < 60:
            cautions.append("Image blur/low detail can hurt detection and OCR accuracy.")
        if scores.get("resolution", 100) < 60:
            cautions.append("Resolution is low for detection/segmentation; consider higher-quality captures.")

    elif modality == "text":
        if overall >= 85 and scores.get("validity", 0) >= 80 and scores.get("uniqueness", 0) >= 80:
            recommendations.extend([
                "NLP/LLM training or fine-tuning corpora.",
                "RAG knowledge base ingestion.",
                "Semantic search and advanced summarization.",
            ])
        elif overall >= 70:
            recommendations.extend([
                "Text classification and topic modeling.",
                "Search index ingestion with light cleaning.",
                "Summarization workflows.",
            ])
        elif overall >= 55:
            recommendations.extend([
                "Keyword extraction and exploratory NLP.",
                "Prototype-level search and QA systems.",
                "Data cleaning and normalization pipelines.",
            ])
        else:
            recommendations.extend([
                "Preprocessing/normalization only before any model use.",
                "Manual curation for quality recovery.",
            ])

        if scores.get("language_consistency", 100) < 60:
            cautions.append("Language/noise consistency is low; normalize encoding and text cleaning first.")

    return recommendations, cautions


def _improvement_actions(modality: str, weaknesses: List[Dict[str, Any]]) -> List[str]:
    action_map = {
        "completeness": "Backfill missing values, enforce ingestion required fields, and add null-rate alerts.",
        "validity": "Add schema validation and type checks at ingest (ranges, formats, enums).",
        "consistency": "Normalize formats and datatypes (dates, booleans, categorical spellings).",
        "uniqueness": "Deduplicate records/assets and define stable unique keys.",
        "integrity": "Investigate duplicate/corrupt entries and improve source reliability.",
        "outliers": "Define outlier policy (winsorization, robust scaling, exclusion rules).",
        "representativeness": "Increase sample diversity/coverage and monitor drift by segment.",
        "timeliness": "Refresh data source cadence and validate event-time recency windows.",
        "balance": "Use stratified sampling or class reweighting to improve class balance.",
        "resolution": "Collect higher-resolution images or avoid heavy downsampling.",
        "sharpness": "Reject blurry captures and enforce camera focus quality gates.",
        "exposure": "Standardize image capture lighting and auto-exposure settings.",
        "contrast": "Use histogram equalization/contrast normalization during preprocessing.",
        "deduplication": "Run perceptual-hash deduplication before train/validation split.",
        "length_quality": "Filter ultra-short/ultra-long documents and chunk long text consistently.",
        "language_consistency": "Normalize encoding/language and remove noisy non-text artifacts.",
    }

    actions: List[str] = []
    for item in weaknesses:
        metric = item.get("metric")
        if metric in action_map:
            actions.append(action_map[metric])

    if modality == "text":
        actions.append("Detect and redact PII where required by policy before model training.")

    # Keep actions unique but ordered.
    seen = set()
    unique_actions = []
    for a in actions:
        if a not in seen:
            unique_actions.append(a)
            seen.add(a)
    return unique_actions[:5]


def assess_data_quality(
    data: Any,
    modality: str = "auto",
    target_col: Optional[str] = None,
    datetime_cols: Optional[List[str]] = None,
    labels: Optional[Dict[str, Any]] = None,
    min_image_resolution: Tuple[int, int] = (224, 224),
) -> Dict[str, Any]:
    selected_modality = detect_modality(data, modality)

    if selected_modality == "tabular":
        result = evaluate_tabular(data, target_col=target_col, datetime_cols=datetime_cols)
    elif selected_modality == "image":
        result = evaluate_images(data, labels=labels, min_resolution=min_image_resolution)
    elif selected_modality == "text":
        result = evaluate_text(data)
    else:
        raise ValueError(f"Unsupported modality: {selected_modality}")

    recommendations, cautions = _recommend_use_cases(
        selected_modality,
        result["dimension_scores"],
        result["overall_score"],
    )

    result["best_use_case"] = recommendations[0] if recommendations else "Manual review required"
    result["recommended_use_cases"] = recommendations
    result["cautions"] = cautions
    result["improvement_actions"] = _improvement_actions(selected_modality, result["weaknesses"])
    result["generated_at_utc"] = datetime.now(timezone.utc).isoformat()

    return result


In [ ]:
def show_report(report: Dict[str, Any]) -> pd.DataFrame:
    print(f"Modality: {report['modality']}")
    print(f"Overall Score: {report['overall_score']} / 100")
    print(f"Quality Tier: {report['quality_tier']}")
    print(f"Best Use Case: {report['best_use_case']}")

    print("\nRecommended Use Cases:")
    for i, item in enumerate(report.get("recommended_use_cases", []), 1):
        print(f"  {i}. {item}")

    if report.get("cautions"):
        print("\nCautions:")
        for c in report["cautions"]:
            print(f"  - {c}")

    if report.get("improvement_actions"):
        print("\nPriority Improvement Actions:")
        for a in report["improvement_actions"]:
            print(f"  - {a}")

    score_df = (
        pd.DataFrame(list(report["dimension_scores"].items()), columns=["dimension", "score"])
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )
    print("\nDimension Scores:")
    print(score_df.to_string(index=False))
    return score_df


def save_report(report: Dict[str, Any], output_path: Optional[Union[str, Path]] = None) -> Path:
    base = Path.cwd()
    if (base / "Alpha" / "outputs").exists():
        out_dir = base / "Alpha" / "outputs"
    elif (base / "outputs").exists():
        out_dir = base / "outputs"
    else:
        out_dir = base / "Alpha" / "outputs"
    out_dir.mkdir(parents=True, exist_ok=True)

    if output_path is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = out_dir / f"quality_report_{report['modality']}_{ts}.json"

    output_path = Path(output_path)
    output_path.write_text(json.dumps(report, indent=2, default=_json_default))
    print(f"\nSaved report: {output_path}")
    return output_path


## Example Runs (using your `Alpha/test_files` samples)

These examples run the scorer on:
- `test_medical.csv` (tabular)
- `test_image.png` (image)
- `test_financial.txt` (text)


In [ ]:
BASE_DIR = Path.cwd()
if (BASE_DIR / "Alpha" / "test_files").exists():
    TEST_DIR = BASE_DIR / "Alpha" / "test_files"
elif (BASE_DIR / "test_files").exists():
    TEST_DIR = BASE_DIR / "test_files"
else:
    TEST_DIR = BASE_DIR

print("Using test directory:", TEST_DIR)


In [ ]:
tabular_path = TEST_DIR / "test_medical.csv"
if tabular_path.exists():
    tabular_report = assess_data_quality(tabular_path, modality="tabular", target_col="outcome")
    tabular_scores = show_report(tabular_report)
    save_report(tabular_report)
else:
    print("Tabular example not found:", tabular_path)


In [ ]:
image_path = TEST_DIR / "test_image.png"
if image_path.exists():
    image_report = assess_data_quality(image_path, modality="image")
    image_scores = show_report(image_report)
    save_report(image_report)
else:
    print("Image example not found:", image_path)


In [ ]:
text_path = TEST_DIR / "test_financial.txt"
if text_path.exists():
    text_report = assess_data_quality(text_path, modality="text")
    text_scores = show_report(text_report)
    save_report(text_report)
else:
    print("Text example not found:", text_path)


## How to use with your own data

- **DataFrame**:
```python
report = assess_data_quality(df, modality="tabular", target_col="label")
```

- **CSV/Parquet path**:
```python
report = assess_data_quality("/path/to/data.csv", modality="tabular")
```

- **Image directory**:
```python
report = assess_data_quality("/path/to/images", modality="image")
```

- **List of texts**:
```python
report = assess_data_quality(["doc 1 text", "doc 2 text"], modality="text")
```

- **Auto-detect mode**:
```python
report = assess_data_quality("/path/to/asset", modality="auto")
```
